In [17]:
import pandas as pd

trips = pd.read_csv("./data/trips_small.csv", sep = ',')
cities = pd.read_csv("./data/cities.csv", sep = ',')
bike_types = pd.read_csv("./data/bike_types.csv", sep = ',')
bikes = pd.read_csv("./data/bikes.csv", sep = ',')
stations = pd.read_csv("./data/stations.csv", sep = ',')
station_status = pd.read_csv("./data/stations_status.csv", sep = ',')

## Bike Type

In [18]:
other_cols = bike_types.columns.drop("id")
bike_types["type_info_missing"] = bike_types[other_cols].isna().all(axis=1)

# Copy propulsion_type, and assume "human" for the empty types
bike_types["propulsion_filled"] = bike_types["propulsion_type"]
bike_types.loc[bike_types["type_info_missing"], "propulsion_filled"] = "human"

# Text columns: fill with "unknown"
for col in ["vehicle_image", "name", "description", "form_factor", "propulsion_type"]:
    bike_types[col] = bike_types[col].fillna("unknown")

# Rider capacity: 1 when unknown
bike_types["rider_capacity"] = bike_types["rider_capacity"].fillna(1).astype(int)

# Range and battery: 0 for all human bikes (original and backfilled)
is_human = bike_types["propulsion_filled"] == "human"
for col in ["max_range", "battery_capacity"]:
    bike_types.loc[is_human, col] = 0

In [21]:
# id 238, 240: DOLLY BIKES "e-Dolly" MaxDrive -> "battery 600 Wh 43 V under bench"
bike_types.loc[bike_types["id"].isin([238, 240]), "battery_capacity"] = 600

# id 179, 181: same eHarry description as 121 and 224, which have 630 Wh
bike_types.loc[bike_types["id"].isin([179, 181]), "battery_capacity"] = 630

In [23]:
bike_types.to_csv("data/cleaned/bike_types_clean.csv", index=False)

## Bikes

In [25]:
bikes

,id,bike_type_id,computer_id
0,400481,196,83300002927
1,374044,204,7551171765
2,373778,204,7551140444
3,34749,14,0
4,88359,143,7551052355
...,...,...,...
88431,372994,204,7551151439
88432,372649,204,7551152435
88433,372951,204,7551150930
88434,374346,204,7551171592


# Trips